In [39]:
import numpy as np
import pandas as pd
import sys
import os

sys.path.append('..')

from rl.environment import NexResolveEnv
from rl.action_space import INDEX_TO_ACTION, ACTION_TO_INDEX
from rl.action_masking import get_valid_actions

## 1. Episode Simulation


In [40]:
env = NexResolveEnv('../data/final/rl_ready_dataset.csv')

state = env.reset(ticket_idx=0)

print(f"--- Episode Start ---")
print(f"Initial State (first 10 dims): {state[:10]}")

total_reward = 0
done = False
step_count = 0

while not done and step_count < 10:
    valid_actions = get_valid_actions(state)
    print([INDEX_TO_ACTION[i] for i in valid_actions])

    # RULE BASED
    if state[23] == 1.0:
        clarify_indices = [i for i in valid_actions if "ask_" in INDEX_TO_ACTION[i]]
        action = clarify_indices[0] if clarify_indices else valid_actions[0]

    elif state[31] >= 0.85:
        suggest_indices = [i for i in valid_actions if "suggest" in INDEX_TO_ACTION[i]]
        action = suggest_indices[0] if suggest_indices else valid_actions[0]

    elif state[21] < 0.5:
        clarify_indices = [i for i in valid_actions if "ask_" in INDEX_TO_ACTION[i]]
        action = clarify_indices[0] if clarify_indices else valid_actions[0]

    else:
        route_indices = [i for i in valid_actions if "route" in INDEX_TO_ACTION[i]]
        action = route_indices[0] if route_indices else valid_actions[0]

    # 👇 DO NOT CHANGE BELOW
    next_state, reward, done, info = env.step(action)

    total_reward += reward
    step_count += 1

    print(f"Step {step_count}: Action={INDEX_TO_ACTION[action]}, Reward={reward:.2f}, Done={done}")

    state = next_state

print(f"--- Episode Finished ---")
print(f"Total Steps: {step_count}, Total Reward: {total_reward:.2f}")

--- Episode Start ---
Initial State (first 10 dims): [1.634e+03 1.800e+02 0.000e+00 0.000e+00 1.000e+00 0.000e+00 9.999e-01
 7.200e+01 1.000e+00 0.000e+00]
['route_bug', 'route_ml_module', 'route_build_infra', 'route_docs', 'route_billing', 'route_product', 'escalate_human']
Step 1: Action=route_bug, Reward=-4.00, Done=False
['route_bug', 'route_ml_module', 'route_build_infra', 'route_docs', 'route_billing', 'route_product', 'escalate_human']
Step 2: Action=route_bug, Reward=-4.00, Done=False
['route_bug', 'route_ml_module', 'route_build_infra', 'route_docs', 'route_billing', 'route_product', 'escalate_human']
Step 3: Action=route_bug, Reward=-4.00, Done=False
['route_bug', 'route_ml_module', 'route_build_infra', 'route_docs', 'route_billing', 'route_product', 'escalate_human']
Step 4: Action=route_bug, Reward=-4.00, Done=False
['route_bug', 'route_ml_module', 'route_build_infra', 'route_docs', 'route_billing', 'route_product', 'escalate_human']
Step 5: Action=route_bug, Reward=-4.00, 

In [41]:
env = NexResolveEnv('../data/final/rl_ready_dataset.csv')

state = env.reset(ticket_idx=0)

print(f"--- Episode Start ---")
print(f"Initial State (first 10 dims): {state[:10]}")

total_reward = 0
done = False
step_count = 0


while not done and step_count < 10:

    valid_actions = get_valid_actions(state)

    # RANDOM ACTION (for exploration testing)
    action = np.random.choice(valid_actions)

    next_state, reward, done, info = env.step(action)

    total_reward += reward
    step_count += 1

    print(f"Step {step_count}: Action={INDEX_TO_ACTION[action]}, Reward={reward:.2f}, Done={done}")

    state = next_state

print(f"--- Episode Finished ---")
print(f"Total Steps: {step_count}, Total Reward: {total_reward:.2f}")



--- Episode Start ---
Initial State (first 10 dims): [1.634e+03 1.800e+02 0.000e+00 0.000e+00 1.000e+00 0.000e+00 9.999e-01
 7.200e+01 1.000e+00 0.000e+00]
Step 1: Action=route_build_infra, Reward=-4.00, Done=False
Step 2: Action=route_ml_module, Reward=-4.00, Done=False
Step 3: Action=route_product, Reward=-4.00, Done=False
Step 4: Action=route_ml_module, Reward=-4.00, Done=False
Step 5: Action=route_build_infra, Reward=-4.00, Done=False
Step 6: Action=route_billing, Reward=-4.00, Done=False
Step 7: Action=route_billing, Reward=-4.00, Done=False
Step 8: Action=route_ml_module, Reward=-4.00, Done=False
Step 9: Action=route_ml_module, Reward=-4.00, Done=True
--- Episode Finished ---
Total Steps: 9, Total Reward: -36.00


In [42]:
import sys
sys.path.append("..")  # move up to project root

from rl.action_masking import get_action_mask, get_valid_actions
from rl.action_space import INDEX_TO_ACTION

In [43]:
import sys
sys.path.append("..")

from rl.environment import NexResolveEnv
from rl.action_masking import get_valid_actions
from rl.action_space import INDEX_TO_ACTION

env = NexResolveEnv("../data/final/rl_ready_dataset.csv")
df = env.df

for i in range(10):
    state = env.reset(ticket_idx=i)
    valid_actions = get_valid_actions(state)

    row = df.iloc[i]

    print("\n" + "="*40)
    print(f"ISSUE NUMBER: {row['issue_number']}")
    
    # ── TEXT (important for understanding masking) ──
    if "clean_text" in row:
        print("\nTEXT:")
        print(row["clean_text"][:300])  # first 300 chars
    
    # ── LABEL / TYPE ──
    if "primary_label" in row:
        print(f"\nLABEL: {row['primary_label']}")

    # ── CORE FEATURES ──
    print("\nCORE FEATURES:")
    print(f"max_sim              : {round(state[31], 3)}")
    print(f"knowledge_gap_flag   : {state[34]}")
    print(f"needs_clarification  : {state[23]}")
    print(f"completeness_score   : {state[21]}")
    print(f"frustration_level    : {state[27]}")
    print(f"sla_remaining_norm   : {round(state[6], 3)}")
    print(f"urgent_flag          : {state[14]}")
    print(f"tier1_flag           : {state[35]}")
    print(f"tier2_flag           : {state[36]}")

    # ── VALID ACTIONS ──
    print("\nVALID ACTIONS:")
    for a in valid_actions:
        print(f" - {INDEX_TO_ACTION[a]}")

    print("="*40)


ISSUE NUMBER: 303181

TEXT:
rate limiting type: bug constantly hitting rate limiting across all models with simple requests vs code version: code 1.111.0 ( [hash] , 2026-03-06t23:06:10z) os version: linux x64 6.19.8-zen1-1-zen modes: system info |item|value| |cpus|intel(r) core(tm) i7-5600u cpu @ 2.60ghz (4 x 3092)| |gpu statu

LABEL: duplicate

CORE FEATURES:
max_sim              : 0.5139999985694885
knowledge_gap_flag   : 1.0
needs_clarification  : 0.0
completeness_score   : 1.0
frustration_level    : 0.12105000019073486
sla_remaining_norm   : 1.0
urgent_flag          : 0.0
tier1_flag           : 1.0
tier2_flag           : 0.0

VALID ACTIONS:
 - route_bug
 - route_ml_module
 - route_build_infra
 - route_docs
 - route_billing
 - route_product
 - escalate_human

ISSUE NUMBER: 303177

TEXT:
no code change 

LABEL: triage

CORE FEATURES:
max_sim              : 0.41600000858306885
knowledge_gap_flag   : 1.0
needs_clarification  : 1.0
completeness_score   : 1.0
frustration_level    : 0.64

In [44]:
from rl.action_masking import get_action_mask

mask = get_action_mask(state)

print("\nMASK VECTOR:")
print(mask)


MASK VECTOR:
[1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1.]
